# Tutorial 4: Correct wavelength and flux arrays

`pymolfit.correct` accepts either a spectrum file or arrays. Arrays do
not carry a FITS header, so this route uses an `Observation` object to
describe when, where, and how the spectrum was observed.

This example reads a small public HARPS spectrum into NumPy arrays only
to provide reproducible tutorial data. The FITS path is not passed to
the correction. In your own analysis, `wavelength` and `flux` can come
from any source.

## Imports

Install `pymolfit`, `matplotlib`, and `ipympl` in the environment used
by this notebook. Change `%matplotlib widget` to `%matplotlib inline`
if an interactive backend is not available.

In [ ]:
%matplotlib widget

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits

from pymolfit import Observation, correct

## Obtain the arrays

The tutorial table stores wavelength in microns and flux in detector
counts. Only these two columns are extracted.

In [ ]:
candidates = (Path.cwd() / "tutorials", Path.cwd())
TUTORIAL_ROOT = next(
    (path.resolve() for path in candidates if (path / "data").is_dir()),
    None,
)
if TUTORIAL_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook from the PyMolFit repository or tutorials directory"
    )

INPUT = TUTORIAL_ROOT / "data" / "harps_nad_crop_air.fits"

with fits.open(INPUT) as hdul:
    wavelength = np.asarray(hdul["SCIENCE"].data["lambda"], dtype=float)
    flux = np.asarray(hdul["SCIENCE"].data["flux"], dtype=float)

print(f"{wavelength.size} samples")
print(f"{wavelength.min():.6f}-{wavelength.max():.6f} micron")

In [ ]:
valid = np.isfinite(wavelength) & np.isfinite(flux)
scale = np.nanmedian(flux[valid])

plt.figure(figsize=(12, 4))
plt.plot(
    wavelength[valid] * 1e4,
    flux[valid] / scale,
    color="black",
    linewidth=0.7,
)
plt.xlabel("Input barycentric air wavelength [Angstrom]")
plt.ylabel("Flux / median")
plt.title("Input wavelength and flux arrays")
plt.tight_layout()
plt.show()

## Describe the observation

These values normally come from an observing log, an original FITS
header, or the instrument pipeline. `wavelength_frame` describes the
velocity correction already applied to the wavelength array. This
HARPS product is barycentric and records a BERV of about -7.53 km/s.

`wavelength_medium`, supplied later to `correct`, is a different
property: it declares whether the numbers are air or vacuum
wavelengths.

In [ ]:
observation = Observation(
    time="2017-04-06T00:05:02.828",
    latitude_deg=-29.2584,
    longitude_deg=-70.7345,
    altitude_m=2400.0,
    airmass=0.5 * (1.231 + 1.236),
    resolving_power=115_000,
    wavelength_frame="barycentric",
    frame_velocity_km_s=-7.52937637262916,
    target_ra_deg=86.820684,
    target_dec_deg=-51.06638,
    pressure_hpa=770.2,
    temperature_c=14.25,
    relative_humidity_percent=33.0,
    instrument="HARPS",
)

observation

## Protect the astrophysical Na D feature

The exclusion range is specific to this example. It prevents the broad
stellar/circumstellar Na D absorption from influencing atmospheric and
instrumental parameter estimation. The final transmission correction
is still evaluated in the excluded interval.

In [ ]:
EXCLUDE_RANGES = (
    (0.58875, 0.58996),
)

plt.figure(figsize=(12, 4))
plt.plot(
    wavelength[valid] * 1e4,
    flux[valid] / scale,
    color="black",
    linewidth=0.7,
)
for index, (lower, upper) in enumerate(EXCLUDE_RANGES):
    plt.axvspan(
        lower * 1e4,
        upper * 1e4,
        color="tab:red",
        alpha=0.14,
        label="Excluded from fit" if index == 0 else None,
    )
plt.xlabel("Input barycentric air wavelength [Angstrom]")
plt.ylabel("Flux / median")
plt.title("Exclusion range on the input arrays")
plt.legend()
plt.tight_layout()
plt.show()

## Correct the arrays

This is the complete array-input call. PyMolFit uses the same
radiative-transfer and fitting workflow as the file-input tutorials.

In [ ]:
result = correct(
    wavelength=wavelength,
    flux=flux,
    wavelength_unit="micron",
    wavelength_medium="air",
    observation=observation,
    exclude_ranges=EXCLUDE_RANGES,
)

if not result.success:
    raise RuntimeError(result.message)

## Compare before and after

PyMolFit internally transforms the barycentric air wavelengths to the
observatory vacuum frame before fitting. The comparison below converts
both returned spectra to air wavelengths while keeping them in the
common observatory frame.

In [ ]:
observed = result.spectrum.to_air().to_unit("angstrom")
corrected = result.corrected.to_air().to_unit("angstrom")
valid_result = observed.valid & corrected.valid

figure, axes = plt.subplots(
    2,
    1,
    figsize=(12, 7),
    sharex=True,
    constrained_layout=True,
)
axes[0].plot(
    observed.wavelength[valid_result],
    observed.flux[valid_result] / scale,
    color="black",
    linewidth=0.7,
    label="Observed",
)
axes[0].plot(
    corrected.wavelength[valid_result],
    corrected.flux[valid_result] / scale,
    color="tab:blue",
    linewidth=0.7,
    label="Telluric corrected",
)
axes[0].set_ylabel("Flux / median")
axes[0].set_title("Array correction")
axes[0].legend()

axes[1].plot(
    observed.wavelength,
    result.transmission,
    color="tab:red",
    linewidth=0.7,
)
axes[1].set_xlabel("Observatory-frame air wavelength [Angstrom]")
axes[1].set_ylabel("Transmission")

plt.show()

## Result

The corrected arrays are available as
`result.corrected.wavelength`, `result.corrected.flux`, and
`result.corrected.uncertainty`. Use `save_corrected_txt` for a compact
downstream spectrum or `save_fit_product_ecsv` for the complete fit
product.